In [1]:
import pandas as pd
import numpy as np
import requests
import ratelim
import tenacity
import time
import os
import zipfile, io

In [67]:
WMATA_KEY = "5ec08fcd21744590b1a0f655710c4c91"
MBTA_KEY = '4aa18b9edd1341e383dc8fe55a788cca'
MTA_KEY = '7027c575-3c63-477c-91d8-3d404472e8a0'

wmata_url = "https://api.wmata.com/Bus.svc/json/jBusPositions"
mbta_url = "https://api-v3.mbta.com/vehicles"
mta_url = "http://bustime.mta.info/api/siri/vehicle-monitoring.json"

wmata_headers = {"api_key": WMATA_KEY}
mbta_headers = {"x-api-key": MBTA_KEY}

In [3]:
wmata_routes = ['D40', 'D60', 'D80', 'C53', 'C61', 'D20', 'P40', 'C81', 'P16', 'D24', 'P14', 'C23', 'P20', 'C35']
mbta_routes = ['1', '23', '28', '39', '57', '66', '111', '64', '201', '50', '42', '87', '85', '70']
mta_routes = ['B44', 'B46', 'B41', 'BX12', 'M14A+', 'M23+', 'Q98', 'B32', 'M21', 'BX42', 'M50', 'Q67', 'B84', 'Q42']

In [4]:
wmata_fields = {
    'VehicleID': 'vehicle_id',
    'Lat': 'lat',
    'Lon': 'lon',
    'Deviation': 'deviation',
    'DateTime': 'timestamp',
    'TripID': 'trip_id',
    'RouteID': 'route_id',
    'DirectionNum': 'direction',
    'TripStartTime': 'trip_start_time'
}

In [8]:
path = "/Users/ahmadalexander/Desktop/python/eco590/assignment-1-Ahmad-Alexander/"

In [60]:
wmata = pd.read_csv(path+"data/wmata.csv")
mbta = pd.read_csv(path+"data/mbta.csv")
mta = pd.read_csv(path+"data/mta.csv")

/var/folders/xp/k2h07crd2td3638b98wms8540000gn/T/ipykernel_40376/1032044342.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  mbta = pd.read_csv(path+"data/mbta.csv")


In [44]:


# Fix dtypes + drop dupes
wmata['timestamp'] = pd.to_datetime(wmata['timestamp'])
wmata['trip_start_time'] = pd.to_datetime(wmata['trip_start_time'])
wmata[['vehicle_id', 'trip_id']] = wmata[['vehicle_id', 'trip_id']].astype(str)
wmata = wmata.drop_duplicates(subset=['vehicle_id', 'timestamp']).copy()

wmata['hour'] = wmata['timestamp'].dt.hour
wmata['day_of_week'] = wmata['timestamp'].dt.dayofweek
wmata['peak'] = (wmata['hour'].between(7, 10) | wmata['hour'].between(16, 19)).astype(int)
wmata['service_id'] = wmata['day_of_week'].map({0:9,1:9,2:10,3:9,4:6,5:8,6:2})
wmata['trip_start_timedelta'] = pd.to_timedelta(wmata['trip_start_time'].dt.strftime('%H:%M:%S'))

# Load GTFS
response = requests.get("https://api.wmata.com/gtfs/bus-gtfs-static.zip", headers=wmata_headers)
response.raise_for_status()
z = zipfile.ZipFile(io.BytesIO(response.content))
stop_times = pd.read_csv(z.open('stop_times.txt'))
trips      = pd.read_csv(z.open('trips.txt'))
calendar   = pd.read_csv(z.open('calendar.txt'))

# Filter trips to active service days and relevant routes
day_cols = ['monday','tuesday','wednesday','thursday','friday','saturday','sunday']
active_services = calendar.loc[calendar[day_cols].any(axis=1), 'service_id']
wmata_trips = trips.loc[
    trips['route_id'].isin(wmata_routes) & trips['service_id'].isin(active_services),
    ['trip_id', 'route_id', 'direction_id', 'service_id']
]

# Build scheduled headways from first stop of each trip
scheduled = stop_times.merge(wmata_trips, on='trip_id')
scheduled = scheduled[scheduled['stop_sequence'] == scheduled.groupby('trip_id')['stop_sequence'].transform('min')]
scheduled['departure_time'] = pd.to_timedelta(scheduled['departure_time'])
scheduled = scheduled.sort_values(['route_id', 'direction_id', 'service_id', 'departure_time'])
scheduled['scheduled_headway'] = (
    scheduled.groupby(['route_id', 'direction_id', 'service_id'])['departure_time']
    .diff().dt.total_seconds() / 60
)
scheduled = scheduled[['route_id', 'direction_id', 'service_id', 'departure_time', 'scheduled_headway']]

# Merge, drop unmatched rows, build bunching flag
wmata = wmata.merge(scheduled,
    left_on=['route_id', 'direction', 'service_id', 'trip_start_timedelta'],
    right_on=['route_id', 'direction_id', 'service_id', 'departure_time'],
    how='left'
).dropna(subset=['scheduled_headway'])

wmata['bunching'] = (wmata['deviation'] > wmata['scheduled_headway'] * 0.5).astype(int)
wmata = wmata.drop(columns=['trip_start_timedelta', 'direction_id', 'departure_time'])

wmata.to_csv(path+"data/wmata_clean2.csv", index=False)

In [15]:
wmata

,vehicle_id,lat,lon,deviation,timestamp,trip_id,route_id,direction,trip_start_time,agency,hour,day_of_week,peak,service_id,scheduled_headway,bunching
0,5509,38.900446,-77.021919,1.0,2026-03-31 12:06:16,4927020,D40,1,2026-03-31 11:12:00,WMATA,12,1,0,9,12.0,0
1,5515,38.975230,-77.026732,9.0,2026-03-31 12:06:27,6977020,D40,0,2026-03-31 11:12:00,WMATA,12,1,0,9,12.0,1
2,5522,38.948387,-77.027067,17.0,2026-03-31 12:06:31,25982020,D40,1,2026-03-31 11:24:00,WMATA,12,1,0,9,12.0,1
3,5517,38.972468,-77.027016,-2.0,2026-03-31 12:06:23,4651020,D40,0,2026-03-31 11:24:00,WMATA,12,1,0,9,12.0,0
4,5507,38.939690,-77.024966,2.0,2026-03-31 12:06:33,28529020,D40,0,2026-03-31 11:36:00,WMATA,12,1,0,9,12.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
614670,3315,38.906613,-76.926932,0.0,2026-04-06 22:03:28,29678080,C23,1,2026-04-06 22:00:00,WMATA,22,0,0,9,30.0,0
614671,7017,39.004999,-76.907630,48.0,2026-04-06 22:03:21,30064080,P20,0,2026-04-06 20:30:00,WMATA,22,0,0,9,30.0,1
614672,7015,38.957943,-76.868802,57.0,2026-04-06 22:03:04,14090080,P20,0,2026-04-06 21:00:00,WMATA,22,0,0,9,30.0,1
614673,4775,38.952599,-76.878342,0.0,2026-04-06 22:03:31,7743080,P20,0,2026-04-06 22:00:00,WMATA,22,0,0,9,60.0,0


In [45]:
pd.read_csv(path+"data/wmata_clean.csv").equals(pd.read_csv(path+"data/wmata_clean2.csv"))

True

In [61]:

# Load raw data
mbta = pd.read_csv(path+"data/mbta.csv", low_memory=False)

# Filter, fix dtypes, drop nulls and dupes
mbta = mbta[mbta['route_id'].isin([int(r) for r in mbta_routes if r.isdigit()] + mbta_routes)].copy()
mbta['timestamp'] = pd.to_datetime(mbta['timestamp'])
mbta[['route_id', 'trip_id']] = mbta[['route_id', 'trip_id']].astype(str)
mbta['stop_sequence'] = mbta['stop_sequence'].astype('Int64')
mbta = mbta.dropna(subset=['stop_sequence']).drop_duplicates(subset=['vehicle_id', 'timestamp']).copy()

# Time variables
mbta['hour'] = mbta['timestamp'].dt.hour
mbta['day_of_week'] = mbta['timestamp'].dt.dayofweek
mbta['peak'] = (mbta['hour'].between(7, 10) | mbta['hour'].between(16, 19)).astype(int)
mbta['day_type'] = mbta['day_of_week'].map({0:'weekday',1:'weekday',2:'weekday',3:'weekday',4:'friday',5:'saturday',6:'sunday'})
mbta['time_of_day'] = pd.to_timedelta(mbta['timestamp'].dt.strftime('%H:%M:%S'))

# Load GTFS and save locally
response = requests.get("https://cdn.mbta.com/MBTA_GTFS.zip", headers=mbta_headers)
response.raise_for_status()
with open(path+"data/mbta_gtfs.zip", "wb") as f:
    f.write(response.content)
z = zipfile.ZipFile(path+"data/mbta_gtfs.zip")
live_trips      = pd.read_csv(z.open('trips.txt'), low_memory=False)
live_stop_times = pd.read_csv(z.open('stop_times.txt'), low_memory=False)
live_calendar   = pd.read_csv(z.open('calendar.txt'))

# Map day_type to calendar and filter trips to relevant routes
live_calendar['day_type'] = live_calendar['service_id'].apply(
    lambda x: 'saturday' if 'Saturday' in x else 'sunday' if 'Sunday' in x else 'friday' if 'fr1' in x else 'weekday'
)
live_trips_filtered = live_trips[live_trips['route_id'].isin([str(r) for r in mbta_routes])]
live_trips_filtered = live_trips_filtered.merge(live_calendar[['service_id', 'day_type']], on='service_id')

# Build scheduled headways from first stop of each trip
live_scheduled = live_stop_times.merge(live_trips_filtered[['trip_id', 'route_id', 'direction_id', 'day_type']], on='trip_id')
live_scheduled = live_scheduled[live_scheduled['stop_sequence'] == live_scheduled.groupby('trip_id')['stop_sequence'].transform('min')]
live_scheduled = live_scheduled.drop_duplicates(subset=['route_id', 'direction_id', 'day_type', 'departure_time'])
live_scheduled['departure_time'] = pd.to_timedelta(live_scheduled['departure_time'])
live_scheduled = live_scheduled.sort_values(['route_id', 'direction_id', 'day_type', 'departure_time'])
live_scheduled['scheduled_headway'] = (
    live_scheduled.groupby(['route_id', 'direction_id', 'day_type'])['departure_time']
    .diff().dt.total_seconds() / 60
)
live_scheduled = live_scheduled.rename(columns={'direction_id': 'direction'})
live_scheduled = live_scheduled[['route_id', 'direction', 'day_type', 'departure_time', 'scheduled_headway']]

# Nearest-match merge on time of day, drop unmatched
mbta = pd.merge_asof(
    mbta.sort_values('time_of_day'),
    live_scheduled.sort_values('departure_time'),
    left_on='time_of_day',
    right_on='departure_time',
    by=['route_id', 'direction', 'day_type'],
    direction='nearest'
).dropna(subset=['scheduled_headway'])

# Merge scheduled arrival times and calc deviation
live_stop_sched = live_stop_times[['trip_id', 'stop_sequence', 'arrival_time']].copy()
live_stop_sched['arrival_time'] = pd.to_timedelta(live_stop_sched['arrival_time'])

mbta['stop_sequence'] = mbta['stop_sequence'].astype(int)
mbta = mbta.merge(live_stop_sched, on=['trip_id', 'stop_sequence'], how='left')
mbta['deviation'] = (mbta['time_of_day'] - mbta['arrival_time']).dt.total_seconds() / 60
mbta = mbta[(mbta['deviation'] > -60) & (mbta['deviation'] < 60)]

# Bunching flag and cleanup
mbta['bunching'] = (mbta['deviation'] > mbta['scheduled_headway'] * 0.5).astype(int)
mbta = mbta.drop(columns=['time_of_day', 'departure_time', 'arrival_time', 'day_type'])

mbta.to_csv(path+"data/mbta_clean2.csv", index=False)

In [47]:
pd.read_csv(path+"data/mbta_clean.csv").equals(pd.read_csv(path+"data/mbta_clean2.csv"))

/var/folders/xp/k2h07crd2td3638b98wms8540000gn/T/ipykernel_40376/1541151395.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(path+"data/mbta_clean.csv").equals(pd.read_csv(path+"data/mbta_clean2.csv"))
/var/folders/xp/k2h07crd2td3638b98wms8540000gn/T/ipykernel_40376/1541151395.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(path+"data/mbta_clean.csv").equals(pd.read_csv(path+"data/mbta_clean2.csv"))


True

In [48]:

mta = pd.read_csv(path+"data/mta.csv")

# Fix dtypes, filter deviation outliers, drop dupes
mta['timestamp'] = pd.to_datetime(mta['timestamp'], format='mixed')
mta['vehicle_id'] = mta['vehicle_id'].astype(str)
mta = mta[(mta['deviation'] > -60) & (mta['deviation'] < 60)]
mta = mta.drop_duplicates(subset=['vehicle_id', 'timestamp']).copy()

# Time variables
mta['hour'] = mta['timestamp'].dt.hour
mta['day_of_week'] = mta['timestamp'].dt.dayofweek
mta['peak'] = (mta['hour'].between(7, 10) | mta['hour'].between(16, 19)).astype(int)
mta['day_type'] = mta['day_of_week'].map({0:'weekday',1:'weekday',2:'weekday',3:'weekday',4:'weekday',5:'saturday',6:'sunday'})
mta['time_of_day'] = pd.to_timedelta(mta['timestamp'].dt.strftime('%H:%M:%S'))

# Load GTFS — MTA splits by borough, save locally
urls = {
    'brooklyn':  "http://web.mta.info/developers/data/nyct/bus/google_transit_brooklyn.zip",
    'bronx':     "http://web.mta.info/developers/data/nyct/bus/google_transit_bronx.zip",
    'manhattan': "http://web.mta.info/developers/data/nyct/bus/google_transit_manhattan.zip",
    'queens':    "http://web.mta.info/developers/data/nyct/bus/google_transit_queens.zip"
}
zips = {}
for borough, url in urls.items():
    r = requests.get(url)
    r.raise_for_status()
    with open(path+f"data/mta_gtfs_{borough}.zip", "wb") as f:
        f.write(r.content)
    zips[borough] = zipfile.ZipFile(path+f"data/mta_gtfs_{borough}.zip")

# Load and combine trips, stop_times, calendar from all boroughs
mta_trips      = pd.concat([pd.read_csv(z.open('trips.txt')) for z in zips.values()], ignore_index=True)
mta_stop_times = pd.concat([pd.read_csv(z.open('stop_times.txt'), low_memory=False) for z in zips.values()], ignore_index=True)
mta_calendar   = pd.concat([pd.read_csv(z.open('calendar.txt')) for z in zips.values()], ignore_index=True).drop_duplicates()

# Map day_type to calendar and filter trips to relevant routes
mta_calendar['day_type'] = mta_calendar['service_id'].apply(
    lambda x: 'saturday' if 'Saturday' in x else 'sunday' if 'Sunday' in x else 'weekday'
)
mta_trips_filtered = mta_trips[mta_trips['route_id'].isin(mta_routes)]
mta_trips_filtered = mta_trips_filtered.merge(mta_calendar[['service_id', 'day_type']], on='service_id')

# Build scheduled headways from first stop of each trip
mta_scheduled = mta_stop_times.merge(mta_trips_filtered[['trip_id', 'route_id', 'direction_id', 'day_type']], on='trip_id')
mta_scheduled = mta_scheduled[mta_scheduled['stop_sequence'] == mta_scheduled.groupby('trip_id')['stop_sequence'].transform('min')]
mta_scheduled = mta_scheduled.drop_duplicates(subset=['route_id', 'direction_id', 'day_type', 'departure_time'])
mta_scheduled['departure_time'] = pd.to_timedelta(mta_scheduled['departure_time'])
mta_scheduled = mta_scheduled.sort_values(['route_id', 'direction_id', 'day_type', 'departure_time'])
mta_scheduled['scheduled_headway'] = (
    mta_scheduled.groupby(['route_id', 'direction_id', 'day_type'])['departure_time']
    .diff().dt.total_seconds() / 60
)
mta_scheduled = mta_scheduled.rename(columns={'direction_id': 'direction'})
mta_scheduled = mta_scheduled[['route_id', 'direction', 'day_type', 'departure_time', 'scheduled_headway']]

# Nearest-match merge on time of day, drop unmatched
mta = pd.merge_asof(
    mta.sort_values('time_of_day'),
    mta_scheduled.sort_values('departure_time'),
    left_on='time_of_day',
    right_on='departure_time',
    by=['route_id', 'direction', 'day_type'],
    direction='nearest'
).dropna(subset=['scheduled_headway'])

# Bunching flag and cleanup
mta['bunching'] = (mta['deviation'] > mta['scheduled_headway'] * 0.5).astype(int)
mta = mta.drop(columns=['time_of_day', 'departure_time', 'day_type'])

mta.to_csv(path+"data/mta_clean2.csv", index=False)

In [49]:
pd.read_csv(path+"data/mta_clean.csv").equals(pd.read_csv(path+"data/mta_clean2.csv"))

False

In [37]:
a = pd.read_csv(path+"data/mta_clean.csv", low_memory=False)
b = pd.read_csv(path+"data/mta_clean2.csv", low_memory=False)

print(a.shape, b.shape)
print(a.dtypes)
print(b.dtypes)

(643563, 14) (643367, 14)
vehicle_id             int64
lat                  float64
lon                  float64
direction              int64
timestamp             object
route_id              object
call_distance        float64
deviation            float64
agency                object
hour                   int64
day_of_week            int64
peak                   int64
scheduled_headway    float64
bunching               int64
dtype: object
vehicle_id             int64
lat                  float64
lon                  float64
direction              int64
timestamp             object
route_id              object
call_distance        float64
deviation            float64
agency                object
hour                   int64
day_of_week            int64
peak                   int64
scheduled_headway    float64
bunching               int64
dtype: object


In [38]:
print(a['timestamp'].min(), a['timestamp'].max())
print(b['timestamp'].min(), b['timestamp'].max())

2026-03-31 12:04:54.000 2026-04-06 22:03:50.000
2026-03-31 12:04:54.000 2026-04-06 22:03:50.000


In [39]:
a = pd.read_csv(path+"data/mta_clean.csv", low_memory=False)
b = pd.read_csv(path+"data/mta_clean2.csv", low_memory=False)

# find rows in a not in b and vice versa
merged = a.merge(b, on=['vehicle_id', 'timestamp', 'route_id'], how='outer', indicator=True)
print(merged['_merge'].value_counts())

only_in_a = merged[merged['_merge'] == 'left_only']
only_in_b = merged[merged['_merge'] == 'right_only']

print(only_in_a[['vehicle_id', 'timestamp', 'route_id']].head(10))
print(only_in_b[['vehicle_id', 'timestamp', 'route_id']].head(10))


_merge
both          643367
left_only        196
right_only         0
Name: count, dtype: int64
       vehicle_id                timestamp route_id
69644        5522  2026-04-06 01:03:42.432    M14A+
69645        5522  2026-04-06 01:04:42.432    M14A+
69646        5522  2026-04-06 01:06:12.434    M14A+
69647        5522  2026-04-06 01:07:12.434    M14A+
69648        5522  2026-04-06 01:08:12.434    M14A+
69649        5522  2026-04-06 01:09:12.434    M14A+
69650        5522  2026-04-06 01:10:12.434    M14A+
69651        5522  2026-04-06 01:11:42.432    M14A+
69652        5522  2026-04-06 01:12:42.432    M14A+
69653        5522  2026-04-06 01:13:42.432    M14A+
Empty DataFrame
Columns: [vehicle_id, timestamp, route_id]
Index: []


In [40]:
print(mta['deviation'].describe())

count    643367.000000
mean          3.155395
std           7.968159
min         -55.600400
25%          -1.231342
50%           1.529400
75%           5.857058
max          59.979917
Name: deviation, dtype: float64


In [50]:
# Combine all agencies
all_df = pd.concat([wmata, mbta, mta], ignore_index=True)

# Cap scheduled headway at 90 minutes
mbta = mbta[mbta['scheduled_headway'] <= 90].copy()
all_df = pd.concat([wmata, mbta, mta], ignore_index=True)

# Peak vs off-peak bunching by agency
print(all_df.groupby(['agency', 'peak'])['bunching'].mean().unstack())

# Bunching by day of week
print(all_df.groupby(['agency', 'day_of_week'])['bunching'].mean().unstack())

# Bunching by headway bucket
all_df['headway_bucket'] = pd.cut(all_df['scheduled_headway'], bins=[0, 10, 20, 30, 90], labels=['<10min', '10-20min', '20-30min', '30+min'])
print(all_df.groupby(['agency', 'headway_bucket'], observed=True)['bunching'].mean().unstack())

# Deviation by direction and peak
print(all_df.groupby(['agency', 'direction', 'peak'])['deviation'].mean().unstack())

# Route-level bunching
print(all_df.groupby(['agency', 'route_id'])['bunching'].mean().sort_values(ascending=False).groupby(level=0).head(5))
print(all_df.groupby(['agency', 'route_id'])['bunching'].mean().sort_values(ascending=True).groupby(level=0).head(5))



peak           0         1
agency                    
MBTA    0.172099  0.208183
MTA     0.288676  0.303147
WMATA   0.184710  0.221625
day_of_week         0         1         2        3         4         5  \
agency                                                                   
MBTA         0.262808       NaN       NaN      NaN       NaN       NaN   
MTA          0.280709  0.353318  0.360249  0.25746  0.263539  0.353106   
WMATA        0.142242  0.294218  0.243564  0.18107  0.255901  0.197218   

day_of_week         6  
agency                 
MBTA         0.075443  
MTA          0.216637  
WMATA        0.124693  
headway_bucket    <10min  10-20min  20-30min    30+min
agency                                                
MBTA            0.291688  0.103459  0.050459  0.010873
MTA             0.394211  0.182477  0.084272  0.044073
WMATA           0.364132  0.163109  0.038092  0.021474
peak                     0         1
agency direction                    
MBTA   0         -1.02138

In [51]:
print(mbta['day_of_week'].value_counts().sort_index())
print(mbta['timestamp'].min(), mbta['timestamp'].max())


day_of_week
0    89830
6    57540
Name: count, dtype: int64
2026-04-05 03:39:38 2026-04-06 22:03:47


In [52]:
mbta_raw = pd.read_csv(path+"data/mbta.csv", low_memory=False)
print(mbta_raw['timestamp'].min(), mbta_raw['timestamp'].max())
print(mbta_raw.shape)

2026-03-31 12:05:59 2026-04-06 22:03:47
(564204, 9)


In [53]:
mbta_raw = pd.read_csv(path+"data/mbta.csv", low_memory=False)
print(mbta_raw['stop_sequence'].isnull().sum())
print(mbta_raw.shape)
mbta_raw['timestamp'] = pd.to_datetime(mbta_raw['timestamp'])
mbta_raw['day_of_week'] = mbta_raw['timestamp'].dt.dayofweek
print(mbta_raw['day_of_week'].value_counts().sort_index())

84
(564204, 9)
day_of_week
0    95434
1    41734
2    97021
3    97164
4    97543
5    72362
6    62946
Name: count, dtype: int64


In [54]:
mbta_raw = pd.read_csv(path+"data/mbta.csv", low_memory=False)
mbta = mbta_raw[mbta_raw['route_id'].isin([int(r) for r in mbta_routes if r.isdigit()] + mbta_routes)].copy()
print("after route filter:", mbta.shape)
mbta['timestamp'] = pd.to_datetime(mbta['timestamp'])
mbta[['route_id', 'trip_id']] = mbta[['route_id', 'trip_id']].astype(str)
mbta['stop_sequence'] = mbta['stop_sequence'].astype('Int64')
mbta = mbta.dropna(subset=['stop_sequence']).drop_duplicates(subset=['vehicle_id', 'timestamp']).copy()
print("after dropna/dedup:", mbta.shape)
mbta['day_of_week'] = pd.to_datetime(mbta['timestamp']).dt.dayofweek
print(mbta['day_of_week'].value_counts().sort_index())

after route filter: (562450, 9)
after dropna/dedup: (556305, 9)
day_of_week
0    94022
1    41127
2    95459
3    95794
4    96149
5    71493
6    62261
Name: count, dtype: int64


In [55]:
mbta['hour'] = mbta['timestamp'].dt.hour
mbta['day_of_week'] = mbta['timestamp'].dt.dayofweek
mbta['peak'] = (mbta['hour'].between(7, 10) | mbta['hour'].between(16, 19)).astype(int)
mbta['day_type'] = mbta['day_of_week'].map({0:'weekday',1:'weekday',2:'weekday',3:'weekday',4:'friday',5:'saturday',6:'sunday'})
mbta['time_of_day'] = pd.to_timedelta(mbta['timestamp'].dt.strftime('%H:%M:%S'))

mbta = pd.merge_asof(
    mbta.sort_values('time_of_day'),
    live_scheduled.sort_values('departure_time'),
    left_on='time_of_day',
    right_on='departure_time',
    by=['route_id', 'direction', 'day_type'],
    direction='nearest'
).dropna(subset=['scheduled_headway'])
print("after merge_asof:", mbta.shape)
print(mbta['day_of_week'].value_counts().sort_index())

mbta = mbta.merge(live_stop_sched, on=['trip_id', 'stop_sequence'], how='left')
mbta['deviation'] = (mbta['time_of_day'] - mbta['arrival_time']).dt.total_seconds() / 60
mbta = mbta[(mbta['deviation'] > -60) & (mbta['deviation'] < 60)]
print("after deviation filter:", mbta.shape)
print(mbta['day_of_week'].value_counts().sort_index())

after merge_asof: (437295, 16)
day_of_week
0    89885
1    41127
2    91013
3    91230
5    66458
6    57582
Name: count, dtype: int64
after deviation filter: (147370, 18)
day_of_week
0    89830
6    57540
Name: count, dtype: int64


In [56]:
print(mbta['arrival_time'].isnull().sum())
print(mbta['deviation'].describe())
print(mbta['deviation'].isnull().sum())

0
count    147370.000000
mean         -0.635773
std           8.458172
min         -59.950000
25%          -3.450000
50%          -0.016667
75%           3.300000
max          59.516667
Name: deviation, dtype: float64
0


In [57]:
mbta = mbta.merge(live_stop_sched, on=['trip_id', 'stop_sequence'], how='left')
mbta['deviation'] = (mbta['time_of_day'] - mbta['arrival_time']).dt.total_seconds() / 60
print(mbta['deviation'].describe())
print(mbta['deviation'].isnull().sum())

KeyError: 'arrival_time'

In [58]:
print(live_stop_sched.columns.tolist())
print(live_stop_sched.shape)

['trip_id', 'stop_sequence', 'arrival_time']
(2130189, 3)


In [59]:
print(mbta[['trip_id', 'stop_sequence']].dtypes)
print(live_stop_sched[['trip_id', 'stop_sequence']].dtypes)

trip_id          object
stop_sequence     Int64
dtype: object
trip_id          object
stop_sequence     int64
dtype: object


In [62]:
print(mbta.shape)
print(mbta['day_of_week'].value_counts().sort_index())

(147370, 15)
day_of_week
0    89830
6    57540
Name: count, dtype: int64


In [63]:
mbta = mbta.merge(live_stop_sched, on=['trip_id', 'stop_sequence'], how='left')
mbta['deviation'] = (mbta['time_of_day'] - mbta['arrival_time']).dt.total_seconds() / 60
print(mbta.groupby('day_of_week')['deviation'].describe())

KeyError: 'time_of_day'

In [65]:
print(mbta[mbta['day_of_week'].isin([1,2,3,5])]['trip_id'].nunique())
print(live_stop_sched['trip_id'].nunique())

# do any trip_ids from those days exist in live_stop_sched?
days_1235 = mbta[mbta['day_of_week'].isin([1,2,3,5])]['trip_id'].unique()
print(pd.Series(days_1235).isin(live_stop_sched['trip_id']).sum())

0
89578
0


In [69]:
response = requests.get("https://cdn.mbta.com/archive/MBTA_GTFS.zip", headers=mbta_headers)
print(response.status_code)

403


In [72]:


# Load raw data
mbta = pd.read_csv(path+"data/mbta.csv", low_memory=False)

# Filter, fix dtypes, drop nulls and dupes
mbta = mbta[mbta['route_id'].isin([int(r) for r in mbta_routes if r.isdigit()] + mbta_routes)].copy()
mbta['timestamp'] = pd.to_datetime(mbta['timestamp'])
mbta[['route_id', 'trip_id']] = mbta[['route_id', 'trip_id']].astype(str)
mbta['stop_sequence'] = mbta['stop_sequence'].astype('Int64')
mbta = mbta.dropna(subset=['stop_sequence']).drop_duplicates(subset=['vehicle_id', 'timestamp']).copy()

# Time variables
mbta['hour'] = mbta['timestamp'].dt.hour
mbta['day_of_week'] = mbta['timestamp'].dt.dayofweek
mbta['peak'] = (mbta['hour'].between(7, 10) | mbta['hour'].between(16, 19)).astype(int)
mbta['day_type'] = mbta['day_of_week'].map({0:'weekday',1:'weekday',2:'weekday',3:'weekday',4:'friday',5:'saturday',6:'sunday'})
mbta['time_of_day'] = pd.to_timedelta(mbta['timestamp'].dt.strftime('%H:%M:%S'))

# Load archived GTFS matching collection period (March 31 - April 6 2026)
response = requests.get("https://cdn.mbtace.com/archive/20260401.zip", headers=mbta_headers)
response.raise_for_status()
with open(path+"data/mbta_gtfs.zip", "wb") as f:
    f.write(response.content)
z = zipfile.ZipFile(path+"data/mbta_gtfs.zip")
live_trips      = pd.read_csv(z.open('trips.txt'), low_memory=False)
live_stop_times = pd.read_csv(z.open('stop_times.txt'), low_memory=False)
live_calendar   = pd.read_csv(z.open('calendar.txt'))

# Map day_type to calendar and filter trips to relevant routes
live_calendar['day_type'] = live_calendar['service_id'].apply(
    lambda x: 'saturday' if 'Saturday' in x else 'sunday' if 'Sunday' in x else 'friday' if 'fr1' in x else 'weekday'
)
live_trips_filtered = live_trips[live_trips['route_id'].isin([str(r) for r in mbta_routes])]
live_trips_filtered = live_trips_filtered.merge(live_calendar[['service_id', 'day_type']], on='service_id')

# Build scheduled headways from first stop of each trip
live_scheduled = live_stop_times.merge(live_trips_filtered[['trip_id', 'route_id', 'direction_id', 'day_type']], on='trip_id')
live_scheduled = live_scheduled[live_scheduled['stop_sequence'] == live_scheduled.groupby('trip_id')['stop_sequence'].transform('min')]
live_scheduled = live_scheduled.drop_duplicates(subset=['route_id', 'direction_id', 'day_type', 'departure_time'])
live_scheduled['departure_time'] = pd.to_timedelta(live_scheduled['departure_time'])
live_scheduled = live_scheduled.sort_values(['route_id', 'direction_id', 'day_type', 'departure_time'])
live_scheduled['scheduled_headway'] = (
    live_scheduled.groupby(['route_id', 'direction_id', 'day_type'])['departure_time']
    .diff().dt.total_seconds() / 60
)
live_scheduled = live_scheduled.rename(columns={'direction_id': 'direction'})
live_scheduled = live_scheduled[['route_id', 'direction', 'day_type', 'departure_time', 'scheduled_headway']]

# Nearest-match merge on time of day, drop unmatched
mbta = pd.merge_asof(
    mbta.sort_values('time_of_day'),
    live_scheduled.sort_values('departure_time'),
    left_on='time_of_day',
    right_on='departure_time',
    by=['route_id', 'direction', 'day_type'],
    direction='nearest'
).dropna(subset=['scheduled_headway'])

# Merge scheduled arrival times and calc deviation
live_stop_sched = live_stop_times[['trip_id', 'stop_sequence', 'arrival_time']].copy()
live_stop_sched['arrival_time'] = pd.to_timedelta(live_stop_sched['arrival_time'])

mbta['stop_sequence'] = mbta['stop_sequence'].astype(int)
mbta = mbta.merge(live_stop_sched, on=['trip_id', 'stop_sequence'], how='left')
mbta['deviation'] = (mbta['time_of_day'] - mbta['arrival_time']).dt.total_seconds() / 60
mbta = mbta[(mbta['deviation'] > -60) & (mbta['deviation'] < 60)]

# Bunching flag and cleanup
mbta['bunching'] = (mbta['deviation'] > mbta['scheduled_headway'] * 0.5).astype(int)
mbta = mbta.drop(columns=['time_of_day', 'departure_time', 'arrival_time', 'day_type'])

mbta.to_csv(path+"data/mbta_clean2.csv", index=False)

In [73]:
print(mbta['day_of_week'].value_counts().sort_index())
print(mbta.shape)

day_of_week
0    90070
1    41122
2    91114
3    91432
4    73655
5    66484
6    57540
Name: count, dtype: int64
(511417, 15)
